In [1]:
import pandas as pd
import numpy as np
from helpers import *
from util import *
from dataset_path import *
from mappers import *
from disease_defs import DISEASE_DEFS

In [2]:
df_visits = pd.read_csv(VISITS_PATH)
df_meds = pd.read_csv(MEDS_PATH)
df_pmh = pd.read_csv(PMH_PATH)
df_labs = pd.read_csv(LABS_PATH)

In [3]:
visits_history = df_visits.copy()
meds_history = df_meds.copy()
pmh_history = df_pmh.copy()

# Demographics

In [4]:
patient_visit_columns = ['subject_id', 'stay_id',
                         'age', 'gender', 'race', 'ethnicity',
                         'intime', 'outtime', 'in_year', 'edregtime', 'edouttime', 'admittime', 'dischtime', 'intime_icu', 'outtime_icu',
                         'ed_los', 'ed_los_hours', 'next_ed_visit_time', 'next_ed_visit_time_diff', 'next_ed_visit_time_diff_days',
                         'time_to_icu_transfer_hours',
                         'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_acuity',
                         'triage_MAP']
df_master = pd.DataFrame(columns=patient_visit_columns)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df_master.head())

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP


In [5]:
basic_column_mapping = {
    'MRN': 'subject_id',
    'CSN': 'stay_id', 
    'Age': 'age',
    'Gender': 'gender',
    'Race': 'race',
    'Ethnicity': 'ethnicity'
}

df_master[list(basic_column_mapping.values())] = df_visits[list(basic_column_mapping.keys())]

triage_mapping = {
    'Triage_Temp': 'triage_temperature',
    'Triage_HR': 'triage_heartrate', 
    'Triage_RR': 'triage_resprate',
    'Triage_SpO2': 'triage_o2sat',
    'Triage_SBP': 'triage_sbp',
    'Triage_DBP': 'triage_dbp'
}

df_master[list(triage_mapping.values())] = df_visits[list(triage_mapping.keys())]

In [6]:
time_column_mapping = {
    'Arrival_time': 'intime',
    'Departure_time': 'outtime', 
    'Roomed_time': 'edregtime',
    'Dispo_time': 'edouttime'
}

for source_col, target_col in time_column_mapping.items():
    df_master[target_col] = adjust_time_by_subtracting_year(df_visits[source_col])

df_master['admittime'] = adjust_time_by_subtracting_year(df_visits[df_visits['ED_dispo'] == 'Inpatient']['Admit_time'])
df_master['intime_icu'] = adjust_time_by_subtracting_year(df_visits[df_visits['ED_dispo'] == 'ICU']['Admit_time'])

In [7]:
hosp_los_timedelta = pd.to_timedelta(df_visits['Hosp_LOS'], unit='D')
derived_time_operations = {
    'dischtime': lambda: df_master['admittime'] + hosp_los_timedelta,
    'outtime_icu': lambda: df_master['intime_icu'] + hosp_los_timedelta,
    'in_year': lambda: df_visits['Arrival_time'].str[:4].astype(int)
}

for col_name, operation in derived_time_operations.items():
    df_master[col_name] = operation()

In [8]:
visit_interval = pd.to_timedelta(df_visits['Hours_to_next_visit'], unit='h')
visit_metrics = {
    'ed_los': round(df_visits['ED_LOS'] / 24, 1),
    'ed_los_hours': df_visits['ED_LOS'],
    'next_ed_visit_time': df_master['intime'] + visit_interval,
    'next_ed_visit_time_diff': df_visits['Hours_to_next_visit'], 
    'next_ed_visit_time_diff_days': round(df_visits['Hours_to_next_visit'] / 24, 1)
}

for col_name, values in visit_metrics.items():
    df_master[col_name] = values

In [9]:
time_diff_to_icu = df_master['intime_icu'] - df_master['intime']
df_master['time_to_icu_transfer_hours'] = round(time_diff_to_icu.dt.total_seconds() / 3600, 2)

df_master['triage_acuity'] = df_visits['Triage_acuity'].str[0].replace('n', np.nan).astype('Int64')
df_master['triage_MAP'] = round((df_master['triage_sbp'] + 2 * df_master['triage_dbp']) / 3, 1)

display(df_master)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7


# Chief Complaint

In [10]:
complaint_dict = {
    "chiefcom_chest_pain": "chest pain|tight chest|pressure chest|angina|retrosternal pain",
    "chiefcom_abdominal_pain": "abdominal pain|abd pain|stomach ache|belly pain|tummy ache",
    "chiefcom_headache": "headache|lightheaded|migraine|cephalgia|pressure head",
    "chiefcom_shortness_of_breath": "breath",
    "chiefcom_back_pain": "back pain|lower back pain|lumbago|upper back pain",
    "chiefcom_cough": "cough|dry cough|productive cough|hacking cough",
    "chiefcom_nausea_vomiting": "nausea|vomit|vomiting|emesis|throwing up",
    "chiefcom_fever_chills": "fever|chill|temperature|feverish|shivering",
    "chiefcom_syncope": "syncope|faint|fainted|blackout|loss of consciousness",
    "chiefcom_dizziness": "dizz|dizzy|vertigo|lightheaded|spinning|off balance"
}

df_master['CC'] = df_visits['CC']
df_master = encode_chief_complaints(df_master, complaint_dict)
display(df_master)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0,FALL,0,0,0,0,0,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",0,0,0,1,0,0,0,0,0,0
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0,"ABNORMAL LAB,GENERAL WEAKNESS",0,0,0,0,0,0,0,0,0,0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7,POSSIBLE DVT,0,0,0,0,0,0,0,0,0,0
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3,URINARY RETENTION,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3,"MULTIPLE COMPLAINTS,SUICIDAL",0,0,0,0,0,0,0,0,0,0
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3,NaN,0,0,0,0,0,0,0,0,0,0
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7,ARM LACERATION,0,0,0,0,0,0,0,0,0,0
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7,ALCOHOL INTOXICATION,0,0,0,0,0,0,0,0,0,0


# Visit, Medication, Comorbidities Data

In [11]:
visits_history['Arrival_time'] = adjust_time_by_subtracting_year(visits_history['Arrival_time'])
meds_history[['Start_date','End_date']] = meds_history[['Start_date','End_date']].apply(adjust_time_by_subtracting_year)
pmh_history['Noted_date'] = adjust_time_by_subtracting_year(pmh_history['Noted_date'])

visits_history['Admit_service'] = visits_history['Admit_service'].astype(str).fillna('')

pmh_history['Code'] = pmh_history['Code'].astype(str)
pmh_history['Code'] = pmh_history['Code'].str.replace('.', '', regex=False)

comorbidity_codes = build_comorbidity_code_dictionary()

visits_history = visits_history.sort_values(by=['MRN', 'Arrival_time']).reset_index(drop=True)
meds_history.set_index('MRN', inplace=True)
pmh_history = pmh_history.sort_values(by=['MRN', 'Noted_date'])


In [12]:
visit_history_columns = [
    'n_ed_30d', 'n_ed_90d', 'n_ed_365d',
    'n_hosp_30d', 'n_hosp_90d', 'n_hosp_365d',
    'n_icu_30d', 'n_icu_90d', 'n_icu_365d'
]
visit_history_mapping = calculate_visit_history(visits_history)

df_master = pd.merge(
    df_master,
    visit_history_mapping,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [13]:
med_history_columns = ['n_med', 'n_medrecon']
med_history_results = calculate_medication_history(visits_history, meds_history)

df_master = pd.merge(
    df_master,
    med_history_results,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [14]:
comorbidity_results = calculate_comorbidities_batch(visits_history, pmh_history, comorbidity_codes)

df_master = pd.merge(
    df_master,
    comorbidity_results,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [15]:
display(df_master)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0,"ABNORMAL LAB,GENERAL WEAKNESS",0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,6,14,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7,POSSIBLE DVT,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3,URINARY RETENTION,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3,"MULTIPLE COMPLAINTS,SUICIDAL",0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

# Scores

In [16]:
add_triage_MAP(df_master)
add_score_REMS(df_master)
add_score_CART(df_master)
add_score_NEWS(df_master)
add_score_NEWS2(df_master)
add_score_MEWS(df_master)
add_score_CCI(df_master)

score_columns = ['stay_id', 'score_CCI', 'score_CART','score_REMS',  'score_NEWS', 'score_NEWS2', 'score_MEWS']
final_score_columns = [col for col in score_columns if col in df_master.columns]

display(df_master)

Variable 'add_triage_MAP' successfully added
Variable 'Score_REMS' successfully added
Variable 'Score_CART' successfully added
Variable 'Score_NEWS' successfully added
Variable 'Score_NEWS2' successfully added
Variable 'Score_MEWS' successfully added
Variable 'add_score_CCI' successfully added


,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.333333,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,3,4,2,2,0,8
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.000000,"ABNORMAL LAB,GENERAL WEAKNESS",0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,6,14,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,3,4,3,3,2,9
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.666667,POSSIBLE DVT,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,10,13,1,1,1,5
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.333333,URINARY RETENTION,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,8,9,1,1,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,N

# Lab Results

In [17]:
df_labs_processed = df_labs.copy()
test_column_name = 'Component_name'
csn_column_name = 'CSN'
value_column_name = 'Component_value'

df_labs_filtered = process_lab_data(df_labs, min_frequency=10000)

df_labs_aggregated = (df_labs_filtered
                      .groupby(['CSN', 'normalized_test'])['Component_value']
                      .mean()
                      .reset_index())

df_labs_wide = (df_labs_aggregated
                .pivot(index='CSN', columns='normalized_test', values='Component_value')
                .reset_index())

df_labs_wide.columns.name = None

df_master = merge_lab_data_with_visits(df_master, df_labs_wide)

print(f"Final dataset shape: {df_master.shape}")
display(df_master)

/Users/xinniemai/Documents/mc-med-benchmark/util.py:281: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)
/Users/xinniemai/Documents/mc-med-benchmark/util.py:281: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)
/Users/xinniemai/Documents/mc-med-benchmark/util.py:281: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)


Found 68 tests with > 10000 occurrences.
Final dataset shape: (118385, 173)


,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)"
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1,34.0,NaN,4.500000,3.016667,22.2,NaN,NaN,1.0200,140.0,17.0,8.1,NaN,NaN
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.333333,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,3,4,2,2,0,8,3.8,290.0,27.0,15.0,35.0,NaN,2.5,44.0,NaN,56.0,NaN,90.0,25.0,1.2100

# Outcome

In [18]:
visits_history_copy = df_visits.copy()

time_cols = ["Arrival_time", "Roomed_time", "Departure_time", "Dispo_time", "Admit_time"]

for col in time_cols:
    year = (
        visits_history_copy[col].str.extract(r"(\d{4})")[0]
        .dropna()
        .astype(int) - 200
    )
    
    decrypted_time = (
        year.astype(str) + 
        visits_history_copy[col].str[4:]
    )

    visits_history_copy[f"{col}_200"] = pd.to_datetime(
        decrypted_time,
        format="%Y-%m-%dT%H:%M:%SZ",  
        errors="coerce" 
    )

In [19]:
pmh_history_copy = df_pmh.copy()

time_cols = ["Noted_date"]

for col in time_cols:
    year = (
        pmh_history_copy[col].str.extract(r"(\d{4})")[0]
        .dropna()
        .astype(int) - 200
    )
    
    decrypted_time = (
        year.astype(str) + 
        pmh_history_copy[col].str[4:]
    )
    
    pmh_history_copy[f"{col}_200"] = pd.to_datetime(
        decrypted_time,
        format="%Y-%m-%dT%H:%M:%SZ", 
        errors="coerce" 
    )

pmh_history_copy["Noted_date_200"] = pd.to_datetime(pmh_history_copy["Noted_date_200"])

In [20]:
pmh_history_copy['icd_version'] = pmh_history_copy['CodeType'].apply(lambda x: 9 if 'Dx9' in x else 10)

pmh_history_copy['ICD_10_Code'] = pmh_history_copy.apply(
    lambda row: convert_9to10(row['Code']) if row['icd_version'] == 9 else row['Code'],
    axis=1
)
pmh_history_copy['icd_version'] = 10

pmh_history_copy["ICD_9_Code"] = pmh_history_copy.apply(
    lambda row: convert_10to9(row["ICD_10_Code"]), axis=1
)

In [21]:
visits_history_copy['Dx_ICD9_list']  = visits_history_copy['Dx_ICD9'].apply(split_clean_icd)
visits_history_copy['Dx_ICD10_list'] = visits_history_copy['Dx_ICD10'].apply(split_clean_icd)

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name'].fillna('')

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name_clean'].str.lower()

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name_clean'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

In [22]:
for disease_name, disease_def in DISEASE_DEFS.items():
    codes = [c.upper() for c in disease_def['codes']]
    keywords = disease_def['keywords']

    match_icd9 = visits_history_copy['Dx_ICD9_list'].apply(lambda lst: any(any(code and c.startswith(code) for code in codes) for c in lst))
    match_icd10 = visits_history_copy['Dx_ICD10_list'].apply(lambda lst: any(any(code and c.startswith(code) for code in codes) for c in lst))

    if keywords:
        keyword_pattern = '|'.join(keywords)
        match_kw = visits_history_copy['Dx_name_clean'].str.contains(keyword_pattern, case=False, na=False, regex=True)
    else:
        match_kw = False

    match_any = match_icd9 | match_icd10 | match_kw

    hit_ids = set(visits_history_copy.loc[match_any, 'CSN'])

    visits_history_copy[f'outcome_{disease_name}'] = visits_history_copy['CSN'].isin(hit_ids).astype(int)

In [23]:
visits_history_copy["outcome_hospitalization"] = (
    visits_history_copy["Admit_time"].notna() & 
    (visits_history_copy["ED_dispo"] == "Inpatient")
).astype(int)
visits_history_copy["outcome_critical"] = (visits_history_copy["ED_dispo"] == "ICU").astype(int)

visits_history_copy["outcome_discharge"] = (visits_history_copy["ED_dispo"] == "Discharge").astype(int)

In [24]:
visits_history_copy["Hours_to_next_visit"] = pd.to_numeric(visits_history_copy["Hours_to_next_visit"])

visits_history_copy["outcome_ed_revisit_72h"] = ((visits_history_copy["Hours_to_next_visit"] < 72)).astype(int)
visits_history_copy["outcome_ed_revisit_168h"] = ((visits_history_copy["Hours_to_next_visit"] < 168)).astype(int)
visits_history_copy["outcome_ed_revisit_720h"] = ((visits_history_copy["Hours_to_next_visit"] < 720)).astype(int)

In [25]:
outcome_cols_regular = [
    'outcome_sepsis',
    'outcome_copd_exac', 
    'outcome_acs_mi',
    'outcome_stroke',
    'outcome_ards',
    'outcome_aki',
    'outcome_pe',
    'outcome_pneumonia_bacterial',
    'outcome_pneumonia_viral',
    'outcome_pneumonia_all',
    'outcome_asthma_acute_exac',
    'outcome_ahf',
    'outcome_hospitalization',
    'outcome_critical',
    'outcome_discharge',
    'outcome_ed_revisit_72h',
    'outcome_ed_revisit_168h',
    'outcome_ed_revisit_720h'
]

existing_regular_cols = [col for col in outcome_cols_regular if col in visits_history_copy.columns]

merge_columns_regular = ['MRN', 'CSN'] + existing_regular_cols

df_master['subject_id'] = df_master['subject_id'].astype(str)
df_master['stay_id'] = df_master['stay_id'].astype(str)
visits_history_copy['MRN'] = visits_history_copy['MRN'].astype(str)
visits_history_copy['CSN'] = visits_history_copy['CSN'].astype(str)

df_master = df_master.merge(
    visits_history_copy[merge_columns_regular],
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

df_master

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1,34.0,NaN,4.500000,3.016667,22.2,NaN,NaN,1.0200,140.0,17.0,8.1,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,20

In [26]:
visits_result = apply_outcome_detection(visits_history_copy, pmh_history_copy, DISEASE_DEFS, method='vectorized')

outcome_cols_3d = [col for col in visits_result.columns if col.startswith("outcome_") and col.endswith("_3d")]

conflicting_3d_cols = [col for col in outcome_cols_3d if col in df_master.columns]
if conflicting_3d_cols:
    for col in conflicting_3d_cols:
        print(f"  - {col}")
    df_master = df_master.drop(columns=conflicting_3d_cols)

merge_columns_3d = ['MRN', 'CSN'] + outcome_cols_3d

visits_result['MRN'] = visits_result['MRN'].astype(str)
visits_result['CSN'] = visits_result['CSN'].astype(str)

df_master = df_master.merge(
    visits_result[merge_columns_3d],
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

display(df_master)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h,outcome_sepsis_3d,outcome_copd_exac_3d,outcome_acs_mi_3d,outcome_stroke_3d,outcome_ards_3d,outcome_aki_3d,outcome_pe_3d,outcome_pneumonia_bacterial_3d,outcome_pneumonia_viral_3d,outcome_pneumonia_all_3d,outcome_asthma_acute_exac_3d,outcome_ahf_3d
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1,34.0,NaN,4.500000,3.0

In [27]:
df_master['outcome_icu_transfer_12h'] = (df_master['time_to_icu_transfer_hours'] <= 12).astype(int)

In [28]:
outcome_types = [
    'sepsis', 'copd_exac', 'acs_mi', 'stroke', 'ards', 'aki', 'pe',
    'pneumonia_bacterial', 'pneumonia_viral', 'pneumonia_all',
    'asthma_acute_exac', 'ahf', 'copd_asthma'
]

for outcome_type in outcome_types:
    ed_outcome = f'outcome_{outcome_type}'
    three_day_outcome = f'outcome_{outcome_type}_3d'
    new_outcome = f'outcome_{outcome_type}_3d_new'

    if ed_outcome in df_master.columns and three_day_outcome in df_master.columns:
        df_master[new_outcome] = (df_master[ed_outcome] + df_master[three_day_outcome]).astype(int)

df_master

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h,outcome_sepsis_3d,outcome_copd_exac_3d,outcome_acs_mi_3d,outcome_stroke_3d,outcome_ards_3d,outcome_aki_3d,outcome_pe_3d,outcome_pneumonia_bacterial_3d,outcome_pneumonia_viral_3d,outcome_pneumonia_all_3d,outcome_asthma_acute_exac_3d,outcome_ahf_3d,outcome_icu_transfer_12h,outcome_sepsis_3d_new,outcome_copd_exac_3d_new,outcome_acs_mi_3d_new,outcome_stroke_3d_new,outcome_ards_3d_new,outcome_aki_3d_new,outcome_pe_3d_new,outcome_pneumonia_bacterial_3d_new,outcome_pneumonia_viral_3d_new,outcome_pneumonia_all_3d_new,outcome_asthma_acute_exac_3d_new,outcome_ahf_3d_new
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [29]:
df_master.to_csv(os.path.join(output_path, 'master_dataset.csv'), index=False)